<a href="https://colab.research.google.com/github/arslaan7861/tts-repo/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Character Voice Generator

Turns a multi-character script into one combined voiceover WAV, each
character in their own configured voice.

This notebook is a thin driver: every cell below is a short call into the
`charvoice` library (see `requirements.md` in the repo for the full spec).
All heavy work -- model loading, synthesis, audio encoding -- runs here on
Colab's GPU; nothing requires your local machine.

Cells are safe to re-run: re-running install/model/mount cells does not
re-download or reload anything already present, and re-running the generate
cell reuses the already-loaded model.

## Cell 1 -- Install
Clone (or update) the repo and install `charvoice` into Colab's own Python,
reusing its pre-installed CUDA PyTorch. This cell intentionally uses plain
Python/shell commands only, never `import charvoice` -- the package does not
exist on this machine until this cell finishes.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Plain stdlib only -- charvoice isn't installed yet, so this cell can't
# import anything from it. This is the one cell allowed to do that.
REPO_URL = "https://github.com/arslaan7861/tts-repo.git"
REPO_DIR = Path("/content/tts-repo")


def _run(cmd):
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


try:
    _run(["nvidia-smi"])
except (subprocess.CalledProcessError, FileNotFoundError):
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU.")

if (REPO_DIR / ".git").is_dir():
    _run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    _run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)])

_run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[colab]", "-q"])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

repo_dir = REPO_DIR
print(f"Repo ready at: {repo_dir}")

## Cell 2 -- Config and model
Load `config.yaml` first (cheap), then fetch/verify the TTS model it names --
fetching needs to know *which* model from the config, so config loads first
even though the spec numbers model setup before config loading.

In [ ]:
from charvoice.config import load_config
from charvoice.colab import fetch_model

config = load_config(repo_dir / "config.yaml")
fetch_model(config.tts.default_engine, config.tts.engine(config.tts.default_engine))
print(f"Using engine: {config.tts.default_engine}")

## Cell 3 -- Mount Google Drive (optional)
Keeps voice profiles, models and output on Drive so they survive a runtime
restart. Each mount also copies in any character folder under `voices/` that
the repo has but Drive doesn't yet -- so adding a new character to the repo
later still reaches Drive on your next mount, without ever touching a
character folder you've already edited there. `scripts/` and `config.yaml`
are copied once, the first time. Skip this cell to stay on the local repo
checkout instead.

In [ ]:
from charvoice.colab import mount_drive, project_dir_for

# repo_dir=repo_dir lets mount_drive seed a first-time Drive folder with the
# repo's voices/scripts/config.yaml -- without it, an empty Drive folder
# would silently have zero voice profiles.
drive_dir = mount_drive(repo_dir=repo_dir)  # returns None if skipped or not in Colab
config = project_dir_for(config, drive_dir)
print(f"Project directory: {config.project_dir}")

## Cell 4 -- Load voice profiles

In [ ]:
from charvoice.profiles import load_profiles

combined = config.paths.voices_combined and config.resolve("voices_combined")
profiles = load_profiles(config.resolve("voices"), combined_file=combined)
print(f"Loaded {len(profiles)} voice profile(s): {', '.join(profiles.ids())}")

## Cell 5 -- Upload the script
Upload a `.txt` file, or paste a script directly into `SCRIPT_TEXT` below and
leave `uploaded` empty.

In [ ]:
from charvoice.parser import parse_script
from charvoice.colab import in_colab

SCRIPT_TEXT = """
MILES: Pete, out of every villain in every universe, which five actually scare you?

PETER1: Five come to mind, but the one that scares me most isn't even the strongest.

MILES: Okay now I have to know. Which one?

PETER1: The one who never stops. Doesn't matter how many times I win, he just shows up again tomorrow.

MILES: That's oddly deep for a guy in a onesie.

PETER1: I contain multitudes, Miles.
"""

if in_colab():
    # from google.colab import files
    # uploaded = files.upload()
    uploaded = None
    if uploaded:
        SCRIPT_TEXT = next(iter(uploaded.values())).decode("utf-8")

lines = parse_script(SCRIPT_TEXT)
print(f"Parsed {len(lines)} line(s).")

## Cell 6 -- Validate
Fails fast, before any GPU time is spent. Reports every unknown speaker at
once, not one at a time.

In [ ]:
from charvoice.pipeline import validate_script

validate_script(lines, profiles)
print("Script is valid.")

## Cell 7 -- Generate the voiceover
The loaded model is held in a process-level cache: re-running this cell
reuses it instead of reloading. Generated segments are also cached on disk,
so a second run over an unchanged script is much faster.

In [ ]:
from charvoice.pipeline import generate_voiceover

def show_progress(p):
    tag = "cached" if p.cached else "generating"
    print(f"[{p.current}/{p.total}] {tag}: {p.speaker}")

result = generate_voiceover(
    lines,
    profiles,
    config,
    overwrite=True,
    progress_cb=show_progress,
)

## Cell 8 -- Summary

In [ ]:
print(result.summary())

## Cell 9 -- Download

In [ ]:
from charvoice.colab import offer_download

offer_download(result.output_path)

### Chatterbox TTS (Quick Test)
Run the cells below to install and test Chatterbox TTS.

In [ ]:
!pip install chatterbox-tts torchaudio

In [ ]:
import torchaudio
from chatterbox.tts import ChatterboxTTS

# Initialize model (Set device to "cuda" for Nvidia GPUs, or "cpu" / "mps" for Mac)
model = ChatterboxTTS.from_pretrained(device="cuda")

# Define text and the reference audio clip you want to clone
text_to_speak = "Pete, out of every villain in every universe, which five actually scare you?"
reference_voice = "voices/miles/reference.wav"

# Generate audio with expression tweaks
wav = model.generate(
    text=text_to_speak,
    audio_prompt_path=reference_voice,
    exaggeration=0.7,
)

# Save the final file locally
torchaudio.save("chatterbox_miles_output.wav", wav, model.sr)
print("Audio generated successfully! Saved to chatterbox_miles_output.wav")

from IPython.display import Audio, display
display(Audio("chatterbox_miles_output.wav"))